## Script de creacion de modelo para produccion

Para este script son necesarios los siguientes ficheros:

1.   requirements.txt
2.   FuncionesTFM.py
3.   Pipeline_limpieza_data.py
4.   Pipeline_selector_variables.py
5.   geo_barcelona.geojson
6.   airbnb_marzo.csv

Nota: Este script fue realizado en Colab por lo que el path de lectura de ficheros incia en  "/content/", se recomienda verificar el path de ubicación si se trabaja en local.


In [ ]:
# Instalar requirements con su versión
!pip install -r "/content/requirements.txt"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.3 MB/s eta 0:00:00
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=da77a99096a99732ff00e62e02b03aeb447102e256299d5248815c84bb80ef40
  Stored in di

In [ ]:
# Librerias
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import sys
import re
import patsy
import ast
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
import torch
import torch.nn as nn
from relativeImp import relativeImp
from  ydata_profiling import ProfileReport
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy.stats import chi2_contingency
import geopandas as gpd
import pickle
import sklearn.impute as skl_imp

from fastapi import FastAPI, UploadFile, File
import io
import requests



In [ ]:
# Funciones
execfile("/content/FuncionesTFM.py")
execfile("/content/Pipeline_limpieza_data.py")
execfile("/content/geo_barcelona.geojson")
execfile("/content/Pipeline_selector_variables.py")
geo = gpd.read_file("geo_barcelona.geojson").to_crs(25831)


In [ ]:
# Data a procesar para crear modelo
data = pd.read_csv('/content/airbnb_marzo.csv')

In [ ]:
#Obtener la data procesada
data_limpia = preparar_limpieza(data)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Columnas y variable objetivo
#obtener lista de variables por selec feats con la funcion seleccionar_variables
variables_finales = seleccionar_variables(data_limpia)
#precio_log como variable objetivo
target_col = 'price_log'

X = data_limpia[variables_finales]
y = data_limpia[target_col]

# 2. Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=123
)

# 3. Definir y entrenar el modelo

modelo_xgb = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=5,
    subsample=0.7,
    colsample_bytree=0.8,
    random_state=123,
    enable_categorical=True,
    n_jobs=-1
)

modelo_xgb.fit(X_train, y_train)

# 4. Evaluar modelo

y_pred = modelo_xgb.predict(X_test)
print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2  :", r2_score(y_test, y_pred))


MAE : 0.2082443690730167
RMSE: 0.2916354552100395
R2  : 0.8840488664120049


In [ ]:
# Guardar en formato  de XGBoost JSON

modelo_xgb.save_model('modelo_xgb_replica.json')


In [ ]:
# leer modelo json
from xgboost import XGBRegressor

modelo_cargado = XGBRegressor()
modelo_cargado.load_model('modelo_xgb_replica.json')
modelo_cargado


XGBRegressor(base_score=[5.059438], booster='gbtree', callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None,
             feature_types=['float', 'float', 'int', 'int', 'float', 'float',
                            'float', 'float', 'float', 'float', 'float',
                            'float', 'float', 'float', 'float', 'flo...
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)